## Analyze Extracted Goals and Source Goal Phrases

This notebook computes statistics from generated goals, including the frequency of goals extracted from interviewer versus stakeholder turns, the number of turns used to extract a goal, etc.

In [1]:
import json

data_path = 'data1_gpt52'

results = json.load(open('%s/extracted-moving.json' % data_path, 'r'))
data = json.load(open('%s/transcripts.json' % data_path, 'r'))

In [2]:
for k, v in results['10'][2].items():
    print('%s: %s' % (k, v))

excerpt: Interviewer: Of course. And if I was to start, my my 1st question to start is like, What is your current role like? What stakeholder role are you? And how does it use your web? Application, choice.

Stakeholder: I'd say I'm a I'm a sort of a average user of Netflix. So my streaming platform of choice is Netflix. I would say, I'm not much of a binge watcher. but I'd say I'm a I'm pretty much an avid user of Netflix. Yeah, I do. I do use it more than the other. Video streaming apps.

Interviewer: So you mentioned that you're an avid user. But just from a context, what do you mean by an avid user like, how many hours per week. Do you use it? Do you use it daily?

Stakeholder: Oh, sure, I don't use it daily, mainly because of like other work and homeworks, but I do use it about 3 to 4Â h per week. Yeah, I would say.

goals: ['Use a video streaming platform for about 3 to 4 hours per week']
phrases: [{'goal': 'Use a video streaming platform for about 3 to 4 hours per week', 'phrase

In [3]:
from collections import Counter
import re

def annotate(excerpt, phrases):
    index = []
    for p in phrases:
        for m in re.finditer(p, excerpt):
            index.append([m.start(), '['])
            index.append([m.end(), ']'])

    
    for i, c in sorted(index, key = lambda x:x[0], reverse = True):
        excerpt = excerpt[:i] + c + excerpt[i:]
    return excerpt

In [4]:


def eval_extracted(extracted, counter=Counter(), base_turn=0):
    errors = []
    review = []
    for e in extracted:
        for g in e['phrases']:
            # if g is not a dict, then model failed to follow format instructions
            if not isinstance(g, dict):
                counter['type errors'] += 1
                continue

            # if g is missing phrases key, then model failed to follow format instructions
            elif not 'phrases' in g:
                counter['type errors'] += 1
                continue

            # if phrase goal doesn't match original generated goal
            if 'goal' not in g:  # added 2026-01-19
                counter['goal mismatch'] += 1
            elif g['goal'] not in e['goals']:
                counter['goal mismatch'] += 1

            # if phrase list is empty
            if len(g['phrases']) == 0:
                counter['no phrases'] += 1

            # check for phrase match separately for interviewer and stakeholder
            i_c_total = 0
            s_c_total = 0
            i_c = 0
            s_c = 0
            turn_match = []
            for p in g['phrases']:
                turns = e['excerpt'].split('\n\n')
                i_c = 0
                s_c = 0
                for i, t in enumerate(turns):
                    i_p = re.findall(r'Interviewer:\s(.+)', t)
                    s_p = re.findall(r'Stakeholder:\s(.+)', t)
                    
                    i_c_t = sum([a.lower().count(p.lower()) for a in i_p])
                    s_c_t = sum([a.lower().count(p.lower()) for a in s_p])
                    if i_c_t + s_c_t > 0:
                        turn_match.append(base_turn + i)

                    i_c += i_c_t
                    s_c += s_c_t

                # if not match between interviewer and stakeholder
                if i_c + s_c == 0:
                    counter['unmatched phrase'] += 1
                    errors.append({'excerpt': e['excerpt'], 'phrase': p, 'goal': g['goal']})

                # if only one match between interviewer and stakeholder
                if i_c + s_c == 1:
                    counter['unique phrases'] += 1
                
                # count speaker role from which matches were found
                counter['from interviewer'] += i_c
                i_c_total += i_c
                s_c_total += s_c
                counter['from stakeholder'] += s_c
                counter['total phrases'] += 1
                
            # if interviewer and stakeholder matched for one goal, then count multi-turn goal
            multi = 'N'
            counter['interviewer goal'] += 1 if i_c > 0 else 0
            counter['stakeholder goal'] += 1 if s_c > 0 else 0
            if i_c_total > 0 and s_c_total > 0:
                counter['multi-turn goal'] += 1
                multi = 'Y'
            if i_c_total + s_c_total > 0:
                review.append({'index': len(review), 'goal': g['goal'] if 'goal' in g else 'missing_goal?', 'excerpt': annotate(e['excerpt'], g['phrases']), 's_match': s_c_total, 'i_match': i_c_total, 'turn_match': turn_match, 'multiturn': multi})
    return counter, errors, review

In [5]:
from collections import Counter
import csv, math

goals = {}
global_count = Counter()
counters = []
match_dist = Counter()
i = 0
for transcript, (key, result) in zip(data['transcript'], results.items()):
    goals[key] = []
    
    for j in range(len(result)):
        counter, errors, review = eval_extracted([result[j]], Counter(), base_turn = j * 2)
        counter['total turns'] = len(transcript)
        counters.append(counter)
        
        for k, c in counter.items():
            global_count[k] += c
    
        with open('%s/cache/%s.%i-generated-goals.csv' % (data_path, i, j), 'w') as f:
            writer = csv.DictWriter(f, fieldnames=['index', 'excerpt', 'goal', 'i_match', 's_match', 'turn_match', 'multiturn'])
            writer.writeheader()
            writer.writerows(review)
    
        # record population data for evaluating the prompt
        goals[key].append([])
        for r in review:
            goals[key][j].append(r['goal'])
            
            for m in r['turn_match']:
                norm_m = math.floor(10 * m / counter['total turns'])
                match_dist[norm_m] += 1

# how does the distribution compare to 'lost in the middle'?
for i in sorted(match_dist.keys()):
    print('%i\t%i' % (i, match_dist[i]))

0	416
1	757
2	909
3	827
4	845
5	718
6	748
7	730
8	519
9	320


In [6]:
fieldnames = set([k for c in counters for k in c.keys()])
with open('%s/statistics.csv' % data_path, 'w') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(counters)

In [7]:
json.dump(goals, open('%s/extracted-goals.json' % data_path, 'w'))

In [8]:
print(global_count)

Counter({'total turns': 133084, 'total phrases': 6947, 'unique phrases': 6758, 'from stakeholder': 6398, 'stakeholder goal': 4067, 'from interviewer': 411, 'interviewer goal': 205, 'multi-turn goal': 183, 'unmatched phrase': 164, 'goal mismatch': 17, 'no phrases': 10})


In [9]:
goals.keys()

dict_keys(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33'])

In [11]:
len(goals['0'])

42

In [14]:
goals['0'][2]

['Use a direct messaging tool that is simple and straightforward',
 'Use the same direct messaging tool that most people around them use']